In [1]:
import osmnx as ox
import pandas as pd

# Data Preparation

In [2]:
area_jaksel = "South Jakarta, Indonesia"
area_jaktim = "East Jakarta, Indonesia"
area_jakpus = "Central Jakarta, Indonesia"
area_jakut = "North Jakarta, Indonesia"
area_jakbar = "West Jakarta, Indonesia"

nama_daerah = [
    "South Jakarta, Indonesia",
    "East Jakarta, Indonesia", 
    "Central Jakarta, Indonesia",
    "North Jakarta, Indonesia",
    "West Jakarta, Indonesia"
]

In [3]:
#boundaries data
df_boundaries = ox.geocode_to_gdf(nama_daerah)
df_boundaries['area_m2'] = df_boundaries.geometry.to_crs(epsg=32748).area
df_boundaries['area_km2'] = df_boundaries['area_m2'] / 1_000_000

df_luas = df_boundaries[['name', 'area_km2']].copy()


In [7]:
#cafe point per district
cafes_jaksel = ox.features_from_place(area_jaksel, tags={"amenity": "cafe"})
cafes_jaktim = ox.features_from_place(area_jaktim, tags={"amenity": "cafe"})
cafes_jakpus = ox.features_from_place(area_jakpus, tags={"amenity": "cafe"})
cafes_jakut = ox.features_from_place(area_jakut, tags={"amenity": "cafe"})
cafes_jakbar = ox.features_from_place(area_jakbar, tags={"amenity": "cafe"})

cafes_jaksel['district'] = 'South Jakarta'
cafes_jaktim['district'] = 'East Jakarta'
cafes_jakpus['district'] = 'Central Jakarta'
cafes_jakbar['district'] = 'West Jakarta'
cafes_jakut['district'] = 'North Jakarta'

df_final = pd.concat([cafes_jaksel, cafes_jaktim, cafes_jakpus, cafes_jakbar, cafes_jakut])

#ambil kolom nama, wilayah, sama geometry aja
df_clean = df_final[['name', 'district', 'geometry']].copy()

#ngehapus kafe yang 'nama' nya kosong
df_clean = df_clean.dropna(subset=['name'])

# 5. Cek hasilnya
print(f"total: {len(df_clean)}")
df_clean.head()

total: 733


name       district  \
element id                                                       
node    1912469553               Cafe Al Tahrir  South Jakarta   
        1912484629  Daily Bread Bakery and Cafe  South Jakarta   
        1912484635                   Kopi Luwak  South Jakarta   
        1912484639   Pandav Coffee;pandav coffe  South Jakarta   
        1912484641                    Starbucks  South Jakarta   

                                      geometry  
element id                                      
node    1912469553  POINT (106.83255 -6.22056)  
        1912484629  POINT (106.83455 -6.21804)  
        1912484635  POINT (106.83526 -6.21825)  
        1912484639  POINT (106.83465 -6.21826)  
        1912484641  POINT (106.83494 -6.21787)

In [6]:
#bypass to get the timestamp data
import time


ox.settings.overpass_endpoint = "https://overpass-api.de/api"
ox.settings.useful_tags_node = list(set(ox.settings.useful_tags_node + ['timestamp']))
ox.settings.useful_tags_way = list(set(ox.settings.useful_tags_way + ['timestamp']))

years = [2022, 2023, 2024, 2025, 2026]
all_years_data = []

for year in years:
    date_param = f'{year}-12-31T23:59:59Z'
    ox.settings.overpass_settings = '[out:json][timeout:300][date:"' + date_param + '"]{maxsize}'
    
    print(f"--- Extracting data {year} ---")
    try:
        tags = {"amenity": "cafe"}
        gdf = ox.features_from_place("Jakarta, Indonesia", tags)
        gdf['snapshot_year'] = year
        all_years_data.append(gdf)
        print(f"Extracted {len(gdf)} point.")
    except Exception as e:
        print(f"Failed extracting {year}: {e}")
    
    time.sleep(5)

#merge all
if all_years_data:
    df_jakarta_growth = pd.concat(all_years_data, ignore_index=True)
    
    # save (optional)
    df_jakarta_growth.to_csv("jakarta_cafe_growth_long.csv", index=False)

# reset the overpass
ox.settings.overpass_settings = '[out:json][timeout:{timeout}]{maxsize}'

--- Extracting data 2022 ---
Extracted 506 point.
--- Extracting data 2023 ---
Extracted 560 point.
--- Extracting data 2024 ---
Extracted 641 point.
--- Extracting data 2025 ---
Extracted 694 point.
--- Extracting data 2026 ---
Extracted 761 point.


In [8]:
# count anual growth
yearly_counts = df_jakarta_growth.groupby('snapshot_year').size().reset_index(name='total_cafes')

print(yearly_counts)

   snapshot_year  total_cafes
0           2022          506
1           2023          560
2           2024          641
3           2025          694
4           2026          761


In [10]:
#make summary
summary = df_clean.groupby('district')['name'].count().reset_index()
summary.columns = ['district', 'total']
summary = summary.sort_values(by='total', ascending=False)

#merge with area
summary = pd.merge(summary, df_luas, left_on='district', right_on='name')
summary = summary.drop(columns=['name'])
summary['density'] = (summary['total'] / summary['area_km2']).round(2)

summary.head()

,district,total,area_km2,density
0,South Jakarta,322,144.719927,2.22
1,West Jakarta,168,126.051236,1.33
2,Central Jakarta,117,48.000451,2.44
3,East Jakarta,65,184.555238,0.35
4,North Jakarta,61,149.377601,0.41


#Map Making

In [11]:
import folium
from folium.plugins import HeatMap, MarkerCluster

heat_data = []
m = folium.Map(location=[-6.2800, 106.8456], zoom_start=11, tiles='CartoDB dark_matter')

#group layer
layer_heatmap = folium.FeatureGroup(name='🔥 Crowd Heatmap')
layer_marker = folium.FeatureGroup(name='📍 Coffee Spots')

#heatmap prep
for index, row in df_clean.iterrows():
    if row.geometry.geom_type == 'Point':
        heat_data.append([row.geometry.y, row.geometry.x])
    elif row.geometry.geom_type == 'Polygon' or row.geometry.geom_type == 'MultiPolygon':
        heat_data.append([row.geometry.centroid.y, row.geometry.centroid.x])
HeatMap(
    heat_data,
    name='🔥 Heatmap Keramaian',
    min_opacity=0.3,
    radius=18,
    blur=25,
    gradient={0.2: '#3d0000', 0.4: '#7a1a00', 0.6: '#c0390b', 0.8: '#e8572a', 1.0: '#ffd700'}
).add_to(layer_heatmap)
layer_heatmap.add_to(m)

In [ ]:
#add boundaries to layer
layer_batas = folium.FeatureGroup(name='🗺️ District Borders')

folium.GeoJson(
    df_boundaries,
    style_function=lambda feature: {
        'fillColor': 'transparent', 
        'color': '#ffffff',        
        'weight': 2,               
        'dashArray': '5, 5'        
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],
        aliases=[' '],
        labels=False,
        sticky=False,
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;")
    )
).add_to(layer_batas)

layer_batas.add_to(m)

In [13]:
#add cafe points
marker_cluster = MarkerCluster(name='📍 Coffee Spots').add_to(m)

warna_map = {
    'South Jakarta': '#FF5722',
    'East Jakarta': '#2196F3',
    'Central Jakarta': '#4CAF50',
    'West Jakarta': '#FFEB3B',
    'North Jakarta': '#9C27B0'
}

for _, row in df_clean.iterrows():
    point = row['geometry'].centroid
    
    # Pop up setting
    popup_html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; min-width: 200px; padding: 8px; background-color: white; border-radius: 8px;">
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <span style="font-size: 18px; margin-right: 8px;">☕</span>
            <b style="color: #E8572A; font-size: 15px;">{row['name']}</b>
    </div>
    """
    
    # Circle maker
    folium.CircleMarker(
        location=[point.y, point.x],
        radius=6,
        color=warna_map.get(row['district'], 'white'),
        fill=True,
        fill_color=warna_map.get(row['district'], 'white'),
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"☕ {row['name']}"
    ).add_to(marker_cluster)

folium.LayerControl(collapsed=False).add_to(m)

m

# Data Visualization

In [14]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#prep your data that you want to visualize
jumlah_kafe = df_clean['district'].value_counts().reset_index()
jumlah_kafe.columns = ['district', 'total']
density = summary[['district', 'density']]

In [15]:
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(
        '🏪 Cafe Count by District',
        '🔥 Cafe Density/km<sup>2</sup>'
    ),
    horizontal_spacing=0.1
)

custom_palette = ['#df5c31', '#ba3f2b', '#8e2b21', '#6f2118', '#5a1d17']
bar_kwargs = dict(
    x=summary['district'],
    marker_color=custom_palette, 
    marker_line_color='rgba(0,0,0,0)',
    opacity=0.9
)

fig.add_trace(
    go.Bar(**bar_kwargs, y=summary['total'],
           name='Total Kafe',
           hovertemplate='<b>%{x}</b><br>%{y} cafe<extra></extra>'), 
    row=1, col=1
)
fig.add_trace(
    go.Bar(**bar_kwargs, y=summary['density'],
           name='Kepadatan',
           hovertemplate='<b>%{x}</b><br>%{y:,} /km<sup>2</sup><extra></extra>'), 
    row=1, col=2
)

fig.update_layout(
    paper_bgcolor='#1a1a1a',
    plot_bgcolor='#242424',
    font_color='#cccccc',
    font_family='"Segoe UI", sans-serif',
    showlegend=False,
    height=450,
    margin=dict(t=90, b=40, l=40, r=40), 
    title=dict(
        text='Jakarta\'s Brewing Statistics',
        font_size=18,
        font_color='#E8572A',
        x=0.5,
        y=0.95 
    )
)

fig.update_xaxes(gridcolor='#333', linecolor='#444')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#333', linecolor='#444')
fig.update_xaxes(showgrid=False)

fig.show()



In [16]:
#graph for annual growth
growth_total = df_jakarta_growth.groupby('snapshot_year').size().reset_index(name='jumlah_kafe')

fig_line_total = px.line(
    growth_total,
    x='snapshot_year',
    y='jumlah_kafe',
    markers=True,
    text='jumlah_kafe',
    hover_data={'snapshot_year': True, 'jumlah_kafe': True},
    labels={
        'snapshot_year': 'Year',
        'jumlah_kafe': 'Total Recorded Cafe'
    },
    title='Annual Growth of Mapped Cafes in Jakarta (2022–2026)'
)

fig_line_total.update_traces(
    line=dict(width=4, color='#E8572A'),
    marker=dict(size=12, opacity=1, line=dict(width=2, color='white')),
    textposition='top center',
    textfont=dict(size=14, color='white', family='Segoe UI')
)
fig_line_total.update_layout(
    paper_bgcolor='#1a1a1a',
    plot_bgcolor='#242424',
    font_color='#cccccc',
    font_family='"Segoe UI", sans-serif',
    height=450,
    margin=dict(t=80, b=40, l=60, r=40),
    title_font_color='#E8572A',
    title_font_size=20,
    title_x=0.5
)

fig_line_total.update_xaxes(
    gridcolor='#333', 
    linecolor='#444', 
    dtick=1,
    title_text='Year'
)
fig_line_total.update_yaxes(
    gridcolor='#333', 
    linecolor='#444',
    title_text='Cafe Count'
)

fig_line_total.show()

In [17]:
# take the html
map_html = m._repr_html_()
chart_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
growth_html = fig_line_total.to_html(full_html=False, include_plotlyjs='cdn')

In [18]:
#another summary for cards
top_wilayah = summary.iloc[0]['district']
total_kafe = df_clean['name'].count()
wilayah_terpadat = summary.loc[summary['density'].idxmax(), 'district']

# HTML Layout

In [21]:
#this dashboard layout developed with the assistance of AI for a clean design

html_template = f"""
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>☕ Jakarta Cafe Dashboard</title>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=Space+Mono:wght@400;700&display=swap" rel="stylesheet">
    <style>
        *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

        :root {{
            --bg: #111111;
            --surface: #1a1a1a;
            --surface2: #242424;
            --border: #2e2e2e;
            --accent: #E8572A;
            --text: #e0e0e0;
            --muted: #888;
        }}

        body {{
            background: var(--bg);
            color: var(--text);
            font-family: 'Plus Jakarta Sans', sans-serif;
            line-height: 1.6;
        }}

        /* ---- Header ---- */
        header {{
            background: var(--surface);
            border-bottom: 1px solid var(--border);
            padding: 28px 40px;
        }}
        .header-inner {{
            max-width: 1200px;
            margin: 0 auto;
            display: flex;
            align-items: flex-end;
            gap: 20px;
            flex-wrap: wrap;
        }}
        .header-title {{
            flex: 1;
        }}
        .header-eyebrow {{
            font-family: 'Space Mono', monospace;
            font-size: 11px;
            color: var(--accent);
            letter-spacing: 3px;
            text-transform: uppercase;
            margin-bottom: 8px;
        }}
        h1 {{
            font-size: clamp(24px, 4vw, 40px);
            font-weight: 700;
            color: white;
            line-height: 1.1;
        }}
        h1 span {{
            color: var(--accent);
        }}
        .header-sub {{
            color: var(--muted);
            font-size: 14px;
            margin-top: 6px;
        }}

        /* ---- Main layout ---- */
        main {{
            max-width: 1200px;
            margin: 0 auto;
            padding: 40px 24px 80px;
        }}

        /* ---- Section label ---- */
        .section-label {{
            font-family: 'Space Mono', monospace;
            font-size: 10px;
            letter-spacing: 3px;
            color: var(--accent);
            text-transform: uppercase;
            margin-bottom: 12px;
        }}
        .section-title {{
            font-size: 20px;
            font-weight: 700;
            color: white;
            margin-bottom: 20px;
        }}

        /* ---- Stats cards ---- */
        .stats-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 16px;
            margin-bottom: 40px;
        }}
        .stat-card {{
            background: var(--surface);
            border: 1px solid var(--border);
            border-radius: 12px;
            padding: 20px;
            transition: border-color 0.2s;
        }}
        .stat-card:hover {{ border-color: var(--accent); }}
        .stat-value {{
            font-size: 32px;
            font-weight: 700;
            color: var(--accent);
            font-family: 'Space Mono', monospace;
            line-height: 1.2;
        }}
        .stat-label {{
            color: var(--muted);
            font-size: 14px;
            margin-top: 4px;
        }}

        /* ---- Card ---- */
        .card {{
            background: var(--surface);
            border: 1px solid var(--border);
            border-radius: 16px;
            overflow: hidden;
            margin-bottom: 32px;
        }}
        .card-header {{
            padding: 20px 24px 0;
        }}
        /* ---- Verdict box ---- */
        .verdict {{
            background: linear-gradient(135deg, rgba(232,87,42,0.12), rgba(232,87,42,0.04));
            border: 1px solid rgba(232,87,42,0.3);
            border-radius: 12px;
            padding: 24px;
            margin-bottom: 32px;
        }}
        .verdict-title {{
            font-size: 16px;
            font-weight: 700;
            color: var(--accent);
            margin-bottom: 8px;
        }}
        .verdict p {{ color: #bbb; font-size: 14px; line-height: 1.7; }}
        /* ---- Map ---- */
        .map-wrapper {{
            height: 520px;
            width: 100%;
        }}
        .map-wrapper iframe, .map-wrapper > div {{
            height: 100% !important;
            width: 100% !important;
            border: none !important;
        }}

        /* ---- Footer ---- */
        footer {{
            border-top: 1px solid var(--border);
            padding: 24px 40px;
            text-align: center;
            color: var(--muted);
            font-size: 13px;
        }}
        footer a {{ color: var(--accent); text-decoration: none; }}
        footer a:hover {{ text-decoration: underline; }}

        @media (max-width: 600px) {{
            header, main {{ padding-left: 16px; padding-right: 16px; }}
            .map-wrapper {{ height: 380px; }}
        }}
    </style>
</head>
<body>

<header>
    <div class="header-inner">
        <div class="header-title">
            <div class="header-eyebrow">Data Visualization · Spatial Analysis</div>
            <h1>The Jakarta <span>Grind</span></h1>
            <p class="header-sub">Spatial exploration of cafe distribution patterns across Jakarta's districts.</p>
        </div>
    </div>
</header>

<main>

    <!-- Stats summary (Hanya Total & Wilayah Terbanyak) -->
    <div class="stats-grid">
        <div class="stat-card">
            <div class="stat-value">{total_kafe}</div>
            <div class="stat-label">OSM Cafe Entries</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{top_wilayah}</div>
            <div class="stat-label">Top District for Cafe-Hoppers</div>
        </div>
        <div class="stat-card">
        <div class="stat-value">{wilayah_terpadat}</div>
        <div class="stat-label">Most Cafe-dense Area</div>
        </div>
    </div>

    <!-- Verdict -->
    <div class="verdict">
        <div class="verdict-title">🔍 Caffeine Check: Jaksel Stereotypes?</div>
        <p>
            <b>Jaksel</b> (South Jakarta) is often labeled as the city\'s <b>trendiest hub</b>, but we\'re letting the coordinates speak the hype. 
            This digital ground check analyzes whether the <b>hits reputation</b> correlates with actual cafe density. 
            Does the data confirm the caffeine boom? Let's check through the <b>spatial lens!</b>
        </p>
    </div>

    <!-- Peta Folium -->
    <div class="card">
        <div class="card-header">
            <div class="section-label">Spatial Visualization</div>
            <div class="section-title">🗺️ Cafe Distribution Map</div>
        </div>
        <div class="map-wrapper">
            {map_html}
        </div>
    </div>
    <!-- Bubble chart -->
    <div class="card">
        <div class="card-header">
            <div class="section-label">Trend Timeline </div>
            <div class="section-title">📈 Charting the Growth: OSM Cafe Data Evolution</div>
        </div>
        <div style="padding: 0 8px 8px;">
            {growth_html}
        </div>
    </div>
    <!-- Grafik Plotly -->
    <div class="card">
        <div class="card-header">
            <div class="section-label">Data Extraction</div>
            <div class="section-title">📊 The Numbers: District Edition</div>
        </div>
        <div style="padding: 0 8px 8px;">
            {chart_html}
        </div>
    </div>

</main>

<footer>
    <p>Powered by OpenStreetMap &bull; Built with Python, Folium & Plotly &bull;
    <a href="https://github.com/aisyahnasywa" target="_blank">View on Github</a></p>
</footer>

</body>
</html>
"""

In [22]:
#save as html
output_file = 'jakarta-cafe-dashboard.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_template)